# QLoRA training

GPU T4. One epoch on CORD train and the 1000 synthetic training invoices, NF4 4-bit base, fp16, LoRA r=16. Logs to W&B and uploads the adapter to a private Hub repo. Training stops if the loss becomes NaN or if the projected time exceeds `MAX_MINUTES`.

Secrets required: `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
import os
import subprocess

REPO = "https://github.com/N-Mehdi/receipt-vlm.git"
DATASET = "N-Mehdi/french-invoices-synth"
ADAPTER = "N-Mehdi/qwen3-vl-2b-receipts-lora"
OUT = "/kaggle/working/outputs"


def run(cmd):
    """Run a shell command, stream its output and fail the notebook on error."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("command failed: " + cmd)


run("git clone --depth 1 " + REPO + " /tmp/receipt-vlm")
os.chdir("/tmp/receipt-vlm")
run("git log -1 --format=%H")
run("pip install -q -r requirements.txt && pip install -q --no-deps -e .")
run(
    'python -c \'import torch, transformers; print("torch", torch.__version__,'
    ' "cuda", torch.version.cuda, "transformers", transformers.__version__)\''
)

In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
os.environ["HF_TOKEN"] = client.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = client.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "receipt-vlm"

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
MAX_MINUTES = 90
run(
    f"python scripts/train.py --invoice-source {DATASET} --output-dir {OUT}/adapter"
    f" --hub-repo {ADAPTER} --max-minutes {MAX_MINUTES}"
)